# Vestigial Gravity: A Phase of Spacetime Without a Frame

**Phase 4 — Vestigial Gravity Simulation (Stakeholder Notebook)**

**Authors:** John Roehm

---

## What is this about?

General relativity describes gravity using a **metric** — a mathematical object that tells you distances and angles in spacetime. But Einstein's theory actually uses a deeper structure called a **tetrad** (or vierbein), which defines a local reference frame at every point.

The metric can be built from the tetrad: $g_{\mu\nu} = \eta_{ab}\, e^a_\mu\, e^b_\nu$. In ordinary general relativity, if you have a tetrad, you automatically get a metric.

But what if spacetime could exist in a state where **the metric exists but the tetrad does not**? This would be a phase of matter where:
- You can measure distances (metric exists)
- You cannot define a local orientation (no preferred frame)

Volovik (2024) called this a **vestigial phase** of gravity, by analogy with vestigial order in condensed matter physics.

## The Three Phases

Our simulation finds three distinct phases as the coupling strength increases:

| Phase | Metric? | Tetrad? | What it means |
|-------|---------|---------|---------------|
| **Pre-geometric** | No | No | No spacetime at all |
| **Vestigial** | Yes | No | Distances exist, but no local frame |
| **Full tetrad** | Yes | Yes | Standard general relativity |

The vestigial phase is the new and interesting one. It sits between "no spacetime" and "full GR" — a partial form of spacetime that has some geometric properties but not all of them.

## Setup

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.core.visualizations import (
    COLORS,
    LAYOUT_TEMPLATE,
    apply_layout,
)
from src.vestigial.mean_field import (
    vestigial_phase_window,
    full_mean_field_analysis,
)
from src.vestigial.phase_diagram import scan_coupling
from src.adw.gap_equation import critical_coupling

LAMBDA = 1.0
N_F = 4
G_c = critical_coupling(LAMBDA, N_F)

## 1. Finding the Three Phases

We scan the coupling strength (think of it as "how strongly the fundamental fermions interact") and measure two quantities:

- **Tetrad signal** ($M_E$): Does a local reference frame form?
- **Metric signal** ($M_g$): Does a distance-measuring geometry form?

When the coupling is weak, neither signal is present (pre-geometric phase). As the coupling increases, the metric signal turns on first, creating the vestigial phase. Only at even stronger coupling does the tetrad also form, giving full general relativity.

In [2]:
# Phase diagram scan
window = vestigial_phase_window(Lambda=LAMBDA, N_f=N_F, n_points=100)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=window["coupling_ratios"],
        y=window["C_tetrad"],
        mode="lines",
        name="Tetrad signal (local frame)",
        line=dict(color=COLORS["Steinhauer"], width=3),
    )
)

fig.add_trace(
    go.Scatter(
        x=window["coupling_ratios"],
        y=window["G_metric"],
        mode="lines",
        name="Metric signal (geometry)",
        line=dict(color=COLORS["Trento"], width=3),
    )
)

# Shade vestigial region
if window["vestigial_exists"]:
    fig.add_vrect(
        x0=window["vestigial_min"],
        x1=window["vestigial_max"],
        fillcolor=COLORS["Trento"],
        opacity=0.12,
        line_width=0,
        annotation_text="Vestigial phase",
        annotation_position="top left",
    )

apply_layout(
    fig,
    title="The Three Phases of Spacetime",
    xaxis_title="Coupling strength (G / G<sub>c</sub>)",
    yaxis_title="Signal strength",
)
fig.show()

The phase diagram shows three distinct regions. At weak coupling (left), the system is pre-geometric — both order parameters are near zero and fluctuations are uncorrelated noise. As coupling increases toward $G_c$, the curvature of the effective potential flattens, and the metric correlation length grows. In the vestigial window ($G/G_c \approx 0.8$--$1.0$), the metric signal becomes collectively ordered while the tetrad remains zero. Above $G_c$, the tetrad condenses and full general relativity emerges.

The key insight: the vestigial phase is not defined by a simple threshold on the metric value, but by the *curvature* at the origin of $V_\mathrm{eff}$ becoming small enough for correlated metric fluctuations to develop.

## 2. Why Does This Matter? The Equivalence Principle

Einstein's Equivalence Principle says that all objects fall the same way in a gravitational field — a feather and a bowling ball dropped in vacuum hit the ground at the same time.

In the vestigial phase, this principle is **violated**. The reason is fundamental:

- **Bosons** (like photons) only need the metric to propagate. They "see" gravity normally in the vestigial phase.
- **Fermions** (like electrons) need the tetrad to define their spin. Without a tetrad, their coupling to gravity is different.

The result: bosons and fermions would follow different paths through the same gravitational field. This is a dramatic, testable prediction.

In [3]:
# EP violation diagnostic
ratios = np.linspace(0.3, 2.5, 80)
ep_values = []
phase_labels = []

for r in ratios:
    result = full_mean_field_analysis(G=r * G_c, Lambda=LAMBDA, N_f=N_F)
    metric_scale = np.sqrt(max(result.G_metric, 1e-30))
    ep = result.C_tetrad / metric_scale if metric_scale > 1e-15 else 0.0
    ep_values.append(min(ep, 1.0))
    phase_labels.append(result.phase)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=ratios,
        y=ep_values,
        mode="lines",
        name="EP compliance",
        line=dict(color=COLORS["Steinhauer"], width=3),
    )
)

fig.add_hline(
    y=1.0,
    line_dash="dot",
    line_color="grey",
    annotation_text="Einstein's EP satisfied",
    annotation_position="top right",
)

if window["vestigial_exists"]:
    fig.add_vrect(
        x0=window["vestigial_min"],
        x1=window["vestigial_max"],
        fillcolor=COLORS["Trento"],
        opacity=0.12,
        line_width=0,
        annotation_text="EP violated",
        annotation_position="top left",
    )

apply_layout(
    fig,
    title="Does the Equivalence Principle Hold?",
    xaxis_title="Coupling strength (G / G<sub>c</sub>)",
    yaxis_title="EP compliance (1 = satisfied, 0 = violated)",
)
fig.show()

In the vestigial phase (amber region), the EP compliance drops to zero: maximum violation. In the full tetrad phase (to the right), the EP is restored — all particles fall the same way, as Einstein predicted.

This provides a clean experimental signature: if we could prepare a system in the vestigial phase, bosonic and fermionic test particles would behave differently in the same gravitational environment.

## 3. The Phase Diagram

The complete phase diagram shows how the three phases are arranged as a function of coupling strength.

In [4]:
# viz-ref: fig_vestigial_phase_diagram

diagram = scan_coupling(
    method="mean_field",
    Lambda=LAMBDA,
    N_f=N_F,
    coupling_range=(0.3, 2.5),
    n_points=50,
)

phase_color_map = {
    "pre_geometric": COLORS["Steinhauer"],
    "vestigial": COLORS["Trento"],
    "full_tetrad": COLORS["Heidelberg"],
}
phase_display = {
    "pre_geometric": "Pre-geometric",
    "vestigial": "Vestigial",
    "full_tetrad": "Full tetrad (GR)",
}

fig = go.Figure()

for phase_name in ["pre_geometric", "vestigial", "full_tetrad"]:
    mask = [p == phase_name for p in diagram.phases]
    x_vals = diagram.coupling_ratios[mask]
    if len(x_vals) > 0:
        fig.add_trace(
            go.Bar(
                x=x_vals,
                y=[1] * len(x_vals),
                name=phase_display[phase_name],
                marker_color=phase_color_map[phase_name],
                width=diagram.coupling_ratios[1] - diagram.coupling_ratios[0],
            )
        )

fig.update_layout(barmode="stack")
apply_layout(
    fig,
    title="Phase Diagram: Three States of Spacetime",
    xaxis_title="Coupling strength (G / G<sub>c</sub>)",
    yaxis_title="",
)
fig.update_yaxes(showticklabels=False)
fig.show()

Reading from left to right:

1. **Pre-geometric** ($G/G_c \lesssim 0.8$): Both order parameters are near zero. The curvature at the origin of $V_\mathrm{eff}$ is large, so fluctuations are short-ranged thermal noise — no collective metric order. No geometry, no frame.

2. **Vestigial** ($0.8 \lesssim G/G_c \lesssim 1.0$): The curvature at the origin has dropped below ~30% of its weak-coupling value. The metric correlation length grows and fluctuations develop collective order: $|M_g| > 0$ while $|M_E| = 0$. Geometry exists without a frame. The Equivalence Principle is violated.

3. **Full tetrad** ($G/G_c > 1.0$): The tetrad condenses ($C_\mathrm{min} > 0$). Both metric and tetrad are present. Standard Einstein-Cartan gravity. EP is restored.

**Key physics:** The vestigial phase is a *curvature-driven* phenomenon — it appears when the restoring force at $C = 0$ weakens enough (near $G_c$) for correlated metric fluctuations to develop, even though the tetrad hasn't yet condensed. This is the same mechanism as vestigial nematic order in frustrated magnets (Fernandes et al. 2019).

## Summary

Our simulation demonstrates that a **vestigial phase of gravity** — spacetime geometry without a local frame — can exist as a stable intermediate phase between pre-geometric chaos and standard general relativity.

**Key findings:**

| Finding | Significance |
|---------|-------------|
| Vestigial phase exists in a finite coupling window | The phase is not fine-tuned |
| Metric has correct (Euclidean) signature | Geometry is well-defined |
| Equivalence Principle is violated | Testable prediction: bosons and fermions behave differently |
| Phase transitions appear continuous | Consistent with second-order transitions |

**What this means for physics:**

If quantum gravity has a vestigial phase, it suggests that the Equivalence Principle is not fundamental — it is an emergent property that appears only when the full tetrad condenses. At the Planck scale, near the pre-geometric/vestigial transition, EP violations could be observable.

**Caveats:**

This is a pilot study using Euclidean (imaginary-time) signature. Extending to physical Lorentzian signature introduces a sign problem that requires more sophisticated simulation techniques. The pilot establishes that the phase structure is present; confirming it in the physical theory is future work.